# Qiskit Backend Selection

This notebook demonstrates how to select different Qiskit backends
when creating an executor via the `Executor` factory.

With alias-based routing, you can pass Qiskit backend strings directly
as the `target` to `Executor.create(...)`, for example `"statevector"`
and `"aer"`.

In [1]:
from executor import Executor, QuantumCircuit, QuantumOperator

## 1. Default backend via plugin name (`qiskit`)

Using the backend plugin name keeps the previous behavior unchanged.

In [2]:
executor_default = Executor.create("qiskit")
print(type(executor_default).__name__)

QiskitExecutor


## 2. Alias strings as direct targets

You can now create Qiskit executors directly via `"statevector"`
or `"aer"` target strings.

In [3]:
executor_statevector = Executor.create("statevector")
print("statevector alias ->", type(executor_statevector).__name__)

try:
    executor_aer = Executor.create("aer")
    print("aer alias ->", type(executor_aer).__name__)
except Exception as exc:
    executor_aer = None
    print("aer alias unavailable in this environment:", exc)

statevector alias -> QiskitExecutor
aer alias -> QiskitExecutor


## 3. Compare expectation values

A Bell-state expectation value computed with different initialization styles.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

observable = QuantumOperator(["ZZ"], [1.0])

result_default = executor_default.expectation_value(qc, observable)
result_statevector = executor_statevector.expectation_value(qc, observable)

print(f"qiskit plugin: {result_default:.6f}")
print(f"statevector alias: {result_statevector:.6f}")
print(f"Difference: {abs(result_default - result_statevector):.2e}")

qiskit plugin: 1.000000
statevector alias: 1.000000
Difference: 0.00e+00


## 4. Backward-compatible explicit style

The previous explicit form remains fully supported.

In [ ]:
executor_explicit = Executor.create("qiskit", backend="statevector")
result_explicit = executor_explicit.expectation_value(qc, observable)
print(f"explicit backend=statevector: {result_explicit:.6f}")

explicit backend=statevector: 1.000000


## 5. Passing a backend instance

Factory auto-detection for backend objects is unchanged.

In [6]:
try:
    from qiskit_aer import AerSimulator

    backend_obj = AerSimulator()
    executor_from_obj = Executor.create(backend_obj)
    print("backend object ->", type(executor_from_obj).__name__)
except Exception as exc:
    print("Backend-object example skipped:", exc)

backend object -> QiskitExecutor


## 6. Sampling with string targets

For shot-based sampling, use an executor configured with shots.

In [7]:
try:
    executor_sample = Executor.create("aer", shots=1000, seed=42)
except Exception:
    # Fallback when qiskit-aer is not available
    executor_sample = Executor.create("qiskit", backend="statevector", shots=1000, seed=42)

samples = executor_sample.sample(qc)
print("Samples:", samples)

Samples: {'00': 482, '11': 518}
